# Scoop Forecast: save a model, serve it in Streamlit
**Assignment 3 — Yaseen Mohamed**

**Business question:** How do temperature and weekend status affect predicted daily ice cream sales?

**New predictor:** `is_weekend` (Saturday/Sunday = 1; other days = 0). My modeling assumption is that weekends attract additional customers at a given temperature. Compare against a temperature-only linear regression.

**Data provenance:** No starter notebook or dataset was supplied. All 730 observations below are **simulated classroom data**, not real shop records. Temperature and weekend effects are intentionally built into the generator. Accuracy here demonstrates the notebook-to-app workflow, not real-world predictive validity or causality.

**Workflow:** generate data → chronological train/test split → evaluate baseline and updated model → refit on all data → save model → reload and verify. The Streamlit app only loads and predicts.

In [1]:
from pathlib import Path
import json
import hashlib
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
(ROOT / 'data').mkdir(exist_ok=True)
(ROOT / 'models').mkdir(exist_ok=True)
FEATURES = ['temperature_f', 'is_weekend']
SEED = 42

## 1. Add the predictor and generate transparent sample data

The same sales target is used for both models. We do not append a random predictor to an unrelated real dataset or call generated values observed sales. Instead, this fully specified simulation assumes:

`sales = 40 + 4 × (temperature_f − 50) + 65 × is_weekend + random noise`

The 65-scoop weekend term is an **assumption** of the simulation. The regressions below learn their own coefficients from training data. We do not hard-code the simulation formula into the web app. Temperature is restricted to 50–100°F, matching the app controls.

In [2]:
rng = np.random.default_rng(SEED)
dates = pd.date_range('2024-01-01', periods=730, freq='D')
seasonal_temperature = 75 + 18 * np.sin(2 * np.pi * (dates.dayofyear.to_numpy() - 100) / 365)
temperature = np.round(np.clip(seasonal_temperature + rng.normal(0, 5, len(dates)), 50, 100), 1)
weekend = (dates.dayofweek >= 5).astype(int)
noise = rng.normal(0, 12, len(dates))
sales = np.maximum(0, np.rint(40 + 4 * (temperature - 50) + 65 * weekend + noise)).astype(int)
df = pd.DataFrame({'date': dates.strftime('%Y-%m-%d'), 'temperature_f': temperature,
                   'is_weekend': weekend, 'sales_scoops': sales})
assert not df.isna().any().any()
assert set(df.is_weekend.unique()) == {0, 1}
df.to_csv(ROOT / 'data' / 'ice_cream_sales.csv', index=False)
print(f'{len(df)} simulated daily observations; target = scoops sold per day')
df.head()

730 simulated daily observations; target = scoops sold per day


,date,temperature_f,is_weekend,sales_scoops
0,2024-01-01,58.7,0,80
1,2024-01-02,51.9,0,48
2,2024-01-03,60.8,0,90
3,2024-01-04,61.8,0,85
4,2024-01-05,50.0,0,43


## 2. Retrain and evaluate on the same held-out days

The first 80% of days train both models; the final 20% evaluate them. No held-out target is used for fitting during evaluation. The updated model uses temperature **and** weekend status. We report mean absolute error (MAE), root mean squared error (RMSE), and R². Lower MAE/RMSE indicates better performance on these simulated test days.

No hyperparameter search or selection is performed on the test set. These data have a stable simulated relationship, so this split is an educational evaluation rather than proof that the model will generalize to actual shops.

In [3]:
split = int(len(df) * 0.8)
train, test = df.iloc[:split].copy(), df.iloc[split:].copy()
baseline = LinearRegression().fit(train[['temperature_f']], train['sales_scoops'])
updated = LinearRegression().fit(train[FEATURES], train['sales_scoops'])

def evaluate(model, columns):
    prediction = model.predict(test[columns])
    return {'mae': float(mean_absolute_error(test.sales_scoops, prediction)),
            'rmse': float(np.sqrt(mean_squared_error(test.sales_scoops, prediction))),
            'r2': float(r2_score(test.sales_scoops, prediction))}

evaluation = {'baseline': evaluate(baseline, ['temperature_f']),
              'updated': evaluate(updated, FEATURES)}
print(f'Train: {len(train)} days ({train.date.iloc[0]} through {train.date.iloc[-1]})')
print(f'Test: {len(test)} days ({test.date.iloc[0]} through {test.date.iloc[-1]})')
pd.DataFrame(evaluation).T.rename(index={'baseline':'Temperature only', 'updated':'Temperature + weekend'})

Train: 584 days (2024-01-01 through 2025-08-06)
Test: 146 days (2025-08-07 through 2025-12-30)


,mae,rmse,r2
Temperature only,26.239947,31.746683,0.699872
Temperature + weekend,10.124730,12.743717,0.951638


## 3. Refit for serving and save the trained model

After recording the evaluation above, fit both models on all 730 days to produce the serving artifact. Held-out scores remain those from step 2; they are **not** scores for this final fit. Save fitted scikit-learn objects and exact feature names with `joblib`. Record data provenance, split dates, learned coefficients and package versions alongside them.

The app loads this trusted bundled artifact with `joblib.load` and calls `.predict`; it does not retrain when someone opens or interacts with it.

In [4]:
final_model = LinearRegression().fit(df[FEATURES], df.sales_scoops)
final_baseline = LinearRegression().fit(df[['temperature_f']], df.sales_scoops)
bundle = {'model': final_model, 'baseline': final_baseline, 'feature_names': FEATURES,
          'target': 'sales_scoops', 'data_kind': 'simulated'}
model_path = ROOT / 'models' / 'ice_cream_model.joblib'
joblib.dump(bundle, model_path, compress=3)
metadata = {
    'data_kind': 'simulated', 'seed': SEED, 'total_rows': len(df),
    'train_rows': len(train), 'test_rows': len(test),
    'split_method': 'chronological 80/20; serving models refit on all rows after evaluation',
    'train_dates': [train.date.iloc[0], train.date.iloc[-1]],
    'test_dates': [test.date.iloc[0], test.date.iloc[-1]],
    'feature_names': FEATURES, 'target': 'sales_scoops', 'evaluation': evaluation,
    'serving_coefficients': dict(zip(FEATURES, final_model.coef_.tolist())),
    'serving_intercept': float(final_model.intercept_),
    'versions': {'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
                 'pandas': pd.__version__, 'numpy': np.__version__, 'joblib': joblib.__version__},
    'model_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(),
    'data_sha256': hashlib.sha256((ROOT / 'data' / 'ice_cream_sales.csv').read_bytes()).hexdigest(),
    'limitation': 'Effects are assumed in simulated data; scores do not establish real-shop accuracy or causality.'
}
(ROOT / 'models' / 'metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
print('Saved:', model_path.relative_to(ROOT))
print('Learned coefficients:', metadata['serving_coefficients'])

Saved: models/ice_cream_model.joblib
Learned coefficients: {'temperature_f': 3.954822140411323, 'is_weekend': 64.57775582101928}


## 4. Reload and verify the app inputs

Predict once for a weekday and once for a weekend at 80°F. This verifies that the added feature survives saving/loading and changes predictions. The feature order is taken from the artifact. `app.py` uses the same DataFrame names and ordering.

In [5]:
reloaded = joblib.load(model_path)
example = pd.DataFrame({'temperature_f': [80.0, 80.0], 'is_weekend': [0, 1]})
predictions = reloaded['model'].predict(example[reloaded['feature_names']])
np.testing.assert_allclose(predictions, final_model.predict(example[FEATURES]))
assert reloaded['model'].n_features_in_ == 2
assert predictions[1] > predictions[0]
print('Reload check passed. App predictions use this saved two-predictor model.')
example.assign(day=['Weekday','Weekend'], predicted_scoops=predictions.round(1))

Reload check passed. App predictions use this saved two-predictor model.


,temperature_f,is_weekend,day,predicted_scoops
0,80.0,0,Weekday,160.1
1,80.0,1,Weekend,224.6


## 5. Use the model in the web app and deploy

Install `requirements.txt`, then run `python -m streamlit run app.py`. The app exposes both predictors, displays predictions from the saved updated model, compares the baseline, and offers this notebook and dataset for download.

Deploy this repository to **Streamlit Community Cloud** with branch `main`, entrypoint `app.py`, and **Python 3.12**. Keep `models/ice_cream_model.joblib`, `models/metadata.json`, `data/ice_cream_sales.csv`, and this notebook in the repository. Runtime versions are pinned to match training. No API key is needed.

**Submission:** the deployed public Streamlit app URL.